**1. Загружаем csv-файл в датафрейм `logs_hotel`:**

In [42]:
import polars as pl

logs_hotel = pl.read_csv("Hotel.csv")

**2. Генерируем второй датафрейм `calendar_dt`:**

In [43]:
start_date = pl.date(2017, 1, 1)
end_date = pl.date(2018, 12, 31)
date_range = pl.date_range(start_date, end_date, interval="1d", eager=True)

calendar_dt = pl.DataFrame({
    "calendar_dt": date_range
})

**3. Выполняем запросы.**

**a)** Среднее количество ночей, которые гости проводят в отеле:

In [44]:
avg_nights = logs_hotel.select(
    (pl.col("weekend_nights") + pl.col("week_nights")).mean().alias("avg_nights")
)

avg_nights_lazy = logs_hotel.lazy().select(
    (pl.col("weekend_nights") + pl.col("week_nights")).mean().alias("avg_nights")
)

print(avg_nights)
print(avg_nights_lazy.collect())

shape: (1, 1)
┌────────────┐
│ avg_nights │
│ ---        │
│ f64        │
╞════════════╡
│ 3.015024   │
└────────────┘
shape: (1, 1)
┌────────────┐
│ avg_nights │
│ ---        │
│ f64        │
╞════════════╡
│ 3.015024   │
└────────────┘


In [45]:
print(avg_nights_lazy.explain())

SELECT [[(col("weekend_nights")) + (col("week_nights"))].mean().alias("avg_nights")]
FROM
  DF ["ID", "n_adults", "n_children", "weekend_nights", ...]; PROJECT["weekend_nights", "week_nights"] 2/19 COLUMNS


В качестве оптимизации можем сперва выбрать нужные столбцы:

In [46]:
avg_nights_optimized = logs_hotel.select(pl.col("weekend_nights"), pl.col("week_nights")).select(
    (pl.col("weekend_nights") + pl.col("week_nights")).mean().alias("avg_nights")
)

print(avg_nights_optimized)

shape: (1, 1)
┌────────────┐
│ avg_nights │
│ ---        │
│ f64        │
╞════════════╡
│ 3.015024   │
└────────────┘


**b)** Процент отменённых броней на каждый месяц:

In [47]:
cancel_percent = logs_hotel.group_by("year", "month").agg(
    (pl.col("status").filter(pl.col("status") == "Canceled").count() / pl.col("status").count() * 100).alias("cancel_percent")
).sort("year", "month")

cancel_percent_lazy = logs_hotel.lazy().group_by("year", "month").agg(
    (pl.col("status").filter(pl.col("status") == "Canceled").count() / pl.col("status").count() * 100).alias("cancel_percent")
).sort("year", "month")

print(cancel_percent)
print(cancel_percent_lazy.collect())

shape: (18, 3)
┌──────┬───────┬────────────────┐
│ year ┆ month ┆ cancel_percent │
│ ---  ┆ ---   ┆ ---            │
│ i64  ┆ i64   ┆ f64            │
╞══════╪═══════╪════════════════╡
│ 2017 ┆ 7     ┆ 66.942149      │
│ 2017 ┆ 8     ┆ 18.244576      │
│ 2017 ┆ 9     ┆ 11.036992      │
│ 2017 ┆ 10    ┆ 15.786722      │
│ 2017 ┆ 11    ┆ 4.173107       │
│ …    ┆ …     ┆ …              │
│ 2018 ┆ 8     ┆ 46.55234       │
│ 2018 ┆ 9     ┆ 45.779878      │
│ 2018 ┆ 10    ┆ 46.357227      │
│ 2018 ┆ 11    ┆ 36.34805       │
│ 2018 ┆ 12    ┆ 18.155757      │
└──────┴───────┴────────────────┘
shape: (18, 3)
┌──────┬───────┬────────────────┐
│ year ┆ month ┆ cancel_percent │
│ ---  ┆ ---   ┆ ---            │
│ i64  ┆ i64   ┆ f64            │
╞══════╪═══════╪════════════════╡
│ 2017 ┆ 7     ┆ 66.942149      │
│ 2017 ┆ 8     ┆ 18.244576      │
│ 2017 ┆ 9     ┆ 11.036992      │
│ 2017 ┆ 10    ┆ 15.786722      │
│ 2017 ┆ 11    ┆ 4.173107       │
│ …    ┆ …     ┆ …              │
│ 2018 ┆ 8     ┆ 4

In [48]:
print(cancel_percent_lazy.explain())

SORT BY [col("year"), col("month")]
  AGGREGATE
    [[([(col("status").filter([(col("status")) == ("Canceled")]).count()) / (col("status").count())]) * (100.0)].alias("cancel_percent")] BY [col("year"), col("month")]
    FROM
    DF ["ID", "n_adults", "n_children", "weekend_nights", ...]; PROJECT["status", "year", "month"] 3/19 COLUMNS


Вновь план отличается лишь проекцией на нужные столбцы.

In [49]:
cancel_percent_optimized = logs_hotel.lazy().select("year", "month", "status").group_by("year", "month").agg(
    (pl.col("status").filter(pl.col("status") == "Canceled").count() / pl.col("status").count() * 100).alias("cancel_percent")
).sort("year", "month")

print(cancel_percent_optimized.collect())

shape: (18, 3)
┌──────┬───────┬────────────────┐
│ year ┆ month ┆ cancel_percent │
│ ---  ┆ ---   ┆ ---            │
│ i64  ┆ i64   ┆ f64            │
╞══════╪═══════╪════════════════╡
│ 2017 ┆ 7     ┆ 66.942149      │
│ 2017 ┆ 8     ┆ 18.244576      │
│ 2017 ┆ 9     ┆ 11.036992      │
│ 2017 ┆ 10    ┆ 15.786722      │
│ 2017 ┆ 11    ┆ 4.173107       │
│ …    ┆ …     ┆ …              │
│ 2018 ┆ 8     ┆ 46.55234       │
│ 2018 ┆ 9     ┆ 45.779878      │
│ 2018 ┆ 10    ┆ 46.357227      │
│ 2018 ┆ 11    ┆ 36.34805       │
│ 2018 ┆ 12    ┆ 18.155757      │
└──────┴───────┴────────────────┘


**c)** Выручка по каждому типу бронирования за каждый месяц:

In [50]:
monthly_revenue = logs_hotel.filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("weekend_nights") + pl.col("week_nights")).alias("total_nights"),
    (pl.col("avg_room_price") * (pl.col("weekend_nights") + pl.col("week_nights"))).alias("booking_revenue")
).group_by("year", "month", "market_segment").agg(
    pl.sum("booking_revenue").alias("total_revenue")
).sort("year", "month", "market_segment")

monthly_revenue_lazy = logs_hotel.lazy().filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("weekend_nights") + pl.col("week_nights")).alias("total_nights"),
    (pl.col("avg_room_price") * (pl.col("weekend_nights") + pl.col("week_nights"))).alias("booking_revenue")
).group_by("year", "month", "market_segment").agg(
    pl.sum("booking_revenue").alias("total_revenue")
).sort("year", "month", "market_segment")

print(monthly_revenue)
print(monthly_revenue_lazy.collect())

shape: (81, 4)
┌──────┬───────┬────────────────┬───────────────┐
│ year ┆ month ┆ market_segment ┆ total_revenue │
│ ---  ┆ ---   ┆ ---            ┆ ---           │
│ i64  ┆ i64   ┆ str            ┆ f64           │
╞══════╪═══════╪════════════════╪═══════════════╡
│ 2017 ┆ 7     ┆ Complementary  ┆ 111.99        │
│ 2017 ┆ 7     ┆ Corporate      ┆ 455.0         │
│ 2017 ┆ 7     ┆ Offline        ┆ 17628.91      │
│ 2017 ┆ 7     ┆ Online         ┆ 9878.98       │
│ 2017 ┆ 8     ┆ Complementary  ┆ 12.0          │
│ …    ┆ …     ┆ …              ┆ …             │
│ 2018 ┆ 11    ┆ Online         ┆ 347977.98     │
│ 2018 ┆ 12    ┆ Complementary  ┆ 100.0         │
│ 2018 ┆ 12    ┆ Corporate      ┆ 15740.44      │
│ 2018 ┆ 12    ┆ Offline        ┆ 90003.65      │
│ 2018 ┆ 12    ┆ Online         ┆ 426696.22     │
└──────┴───────┴────────────────┴───────────────┘
shape: (81, 4)
┌──────┬───────┬────────────────┬───────────────┐
│ year ┆ month ┆ market_segment ┆ total_revenue │
│ ---  ┆ ---   ┆ ---

In [51]:
print(monthly_revenue_lazy.explain())

SORT BY [col("year"), col("month"), col("market_segment")]
  AGGREGATE
    [col("booking_revenue").sum().alias("total_revenue")] BY [col("year"), col("month"), col("market_segment")]
    FROM
     WITH_COLUMNS:
     [[(col("avg_room_price")) * ([(col("weekend_nights")) + (col("week_nights"))].cast(Float64))].alias("booking_revenue")] 
      FILTER [(col("status")) == ("Not_Canceled")]
      FROM
        DF ["ID", "n_adults", "n_children", "weekend_nights", ...]; PROJECT["year", "month", "market_segment", "avg_room_price", ...] 7/19 COLUMNS


In [52]:
monthly_revenue_optimized = logs_hotel.select(["year", "month", "market_segment", "status", "avg_room_price", "weekend_nights", "week_nights"]
).filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("weekend_nights") + pl.col("week_nights")).alias("total_nights"),
    (pl.col("avg_room_price") * (pl.col("weekend_nights") + pl.col("week_nights"))).alias("booking_revenue")
).group_by("year", "month", "market_segment").agg(
    pl.sum("booking_revenue").alias("total_revenue")
).sort("year", "month", "market_segment")

print(monthly_revenue_optimized)

shape: (81, 4)
┌──────┬───────┬────────────────┬───────────────┐
│ year ┆ month ┆ market_segment ┆ total_revenue │
│ ---  ┆ ---   ┆ ---            ┆ ---           │
│ i64  ┆ i64   ┆ str            ┆ f64           │
╞══════╪═══════╪════════════════╪═══════════════╡
│ 2017 ┆ 7     ┆ Complementary  ┆ 111.99        │
│ 2017 ┆ 7     ┆ Corporate      ┆ 455.0         │
│ 2017 ┆ 7     ┆ Offline        ┆ 17628.91      │
│ 2017 ┆ 7     ┆ Online         ┆ 9878.98       │
│ 2017 ┆ 8     ┆ Complementary  ┆ 12.0          │
│ …    ┆ …     ┆ …              ┆ …             │
│ 2018 ┆ 11    ┆ Online         ┆ 347977.98     │
│ 2018 ┆ 12    ┆ Complementary  ┆ 100.0         │
│ 2018 ┆ 12    ┆ Corporate      ┆ 15740.44      │
│ 2018 ┆ 12    ┆ Offline        ┆ 90003.65      │
│ 2018 ┆ 12    ┆ Online         ┆ 426696.22     │
└──────┴───────┴────────────────┴───────────────┘


**d)** Среднее время между бронированием и заездом в каждом месяце:

In [53]:

avg_lead_time = logs_hotel.group_by("year", "month").agg(
    pl.mean("lead_time").alias("avg_lead_time_days")
).sort("year", "month")


avg_lead_time_lazy = logs_hotel.lazy().group_by("year", "month").agg(
    pl.mean("lead_time").alias("avg_lead_time_days")
).sort("year", "month")

print(avg_lead_time)
print(avg_lead_time_lazy.collect())


shape: (18, 3)
┌──────┬───────┬────────────────────┐
│ year ┆ month ┆ avg_lead_time_days │
│ ---  ┆ ---   ┆ ---                │
│ i64  ┆ i64   ┆ f64                │
╞══════╪═══════╪════════════════════╡
│ 2017 ┆ 7     ┆ 146.977961         │
│ 2017 ┆ 8     ┆ 42.250493          │
│ 2017 ┆ 9     ┆ 56.692541          │
│ 2017 ┆ 10    ┆ 66.25196           │
│ 2017 ┆ 11    ┆ 34.425039          │
│ …    ┆ …     ┆ …                  │
│ 2018 ┆ 8     ┆ 115.650947         │
│ 2018 ┆ 9     ┆ 119.367319         │
│ 2018 ┆ 10    ┆ 124.789365         │
│ 2018 ┆ 11    ┆ 82.411916          │
│ 2018 ┆ 12    ┆ 87.509795          │
└──────┴───────┴────────────────────┘
shape: (18, 3)
┌──────┬───────┬────────────────────┐
│ year ┆ month ┆ avg_lead_time_days │
│ ---  ┆ ---   ┆ ---                │
│ i64  ┆ i64   ┆ f64                │
╞══════╪═══════╪════════════════════╡
│ 2017 ┆ 7     ┆ 146.977961         │
│ 2017 ┆ 8     ┆ 42.250493          │
│ 2017 ┆ 9     ┆ 56.692541          │
│ 2017 ┆ 10    ┆ 66.

In [54]:
print(avg_lead_time_lazy.explain())

SORT BY [col("year"), col("month")]
  AGGREGATE
    [col("lead_time").mean().alias("avg_lead_time_days")] BY [col("year"), col("month")]
    FROM
    DF ["ID", "n_adults", "n_children", "weekend_nights", ...]; PROJECT["lead_time", "year", "month"] 3/19 COLUMNS


In [55]:
avg_lead_time_optimized = logs_hotel.select(["year", "month", "lead_time"]).group_by("year", "month").agg(
    pl.mean("lead_time").alias("avg_lead_time_days")
).sort("year", "month")

print(avg_lead_time_optimized)

shape: (18, 3)
┌──────┬───────┬────────────────────┐
│ year ┆ month ┆ avg_lead_time_days │
│ ---  ┆ ---   ┆ ---                │
│ i64  ┆ i64   ┆ f64                │
╞══════╪═══════╪════════════════════╡
│ 2017 ┆ 7     ┆ 146.977961         │
│ 2017 ┆ 8     ┆ 42.250493          │
│ 2017 ┆ 9     ┆ 56.692541          │
│ 2017 ┆ 10    ┆ 66.25196           │
│ 2017 ┆ 11    ┆ 34.425039          │
│ …    ┆ …     ┆ …                  │
│ 2018 ┆ 8     ┆ 115.650947         │
│ 2018 ┆ 9     ┆ 119.367319         │
│ 2018 ┆ 10    ┆ 124.789365         │
│ 2018 ┆ 11    ┆ 82.411916          │
│ 2018 ┆ 12    ┆ 87.509795          │
└──────┴───────┴────────────────────┘


**e)** Общее количество гостей в отеле на каждый день (по убыванию):

In [56]:
daily_guests = logs_hotel.filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("n_adults") + pl.col("n_children")).alias("total_guests")
).group_by("year", "month", "date").agg(
    pl.sum("total_guests").alias("daily_total_guests")
).sort("daily_total_guests", descending=True)

daily_guests_lazy = logs_hotel.lazy().filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("n_adults") + pl.col("n_children")).alias("total_guests")
).group_by("year", "month", "date").agg(
    pl.sum("total_guests").alias("daily_total_guests")
).sort("daily_total_guests", descending=True)

print(daily_guests)
print(daily_guests_lazy.collect())

shape: (538, 4)
┌──────┬───────┬──────┬────────────────────┐
│ year ┆ month ┆ date ┆ daily_total_guests │
│ ---  ┆ ---   ┆ ---  ┆ ---                │
│ i64  ┆ i64   ┆ i64  ┆ i64                │
╞══════╪═══════╪══════╪════════════════════╡
│ 2017 ┆ 8     ┆ 14   ┆ 365                │
│ 2018 ┆ 10    ┆ 13   ┆ 352                │
│ 2017 ┆ 9     ┆ 18   ┆ 345                │
│ 2018 ┆ 12    ┆ 27   ┆ 263                │
│ 2018 ┆ 12    ┆ 8    ┆ 259                │
│ …    ┆ …     ┆ …    ┆ …                  │
│ 2017 ┆ 7     ┆ 23   ┆ 4                  │
│ 2017 ┆ 7     ┆ 9    ┆ 2                  │
│ 2017 ┆ 7     ┆ 13   ┆ 2                  │
│ 2017 ┆ 7     ┆ 3    ┆ 2                  │
│ 2017 ┆ 7     ┆ 22   ┆ 1                  │
└──────┴───────┴──────┴────────────────────┘


shape: (538, 4)
┌──────┬───────┬──────┬────────────────────┐
│ year ┆ month ┆ date ┆ daily_total_guests │
│ ---  ┆ ---   ┆ ---  ┆ ---                │
│ i64  ┆ i64   ┆ i64  ┆ i64                │
╞══════╪═══════╪══════╪════════════════════╡
│ 2017 ┆ 8     ┆ 14   ┆ 365                │
│ 2018 ┆ 10    ┆ 13   ┆ 352                │
│ 2017 ┆ 9     ┆ 18   ┆ 345                │
│ 2018 ┆ 12    ┆ 27   ┆ 263                │
│ 2018 ┆ 12    ┆ 8    ┆ 259                │
│ …    ┆ …     ┆ …    ┆ …                  │
│ 2017 ┆ 7     ┆ 10   ┆ 4                  │
│ 2017 ┆ 7     ┆ 9    ┆ 2                  │
│ 2017 ┆ 7     ┆ 13   ┆ 2                  │
│ 2017 ┆ 7     ┆ 3    ┆ 2                  │
│ 2017 ┆ 7     ┆ 22   ┆ 1                  │
└──────┴───────┴──────┴────────────────────┘


In [57]:
print(daily_guests_lazy.explain())

SORT BY [col("daily_total_guests")]
  AGGREGATE
    [col("total_guests").sum().alias("daily_total_guests")] BY [col("year"), col("month"), col("date")]
    FROM
     WITH_COLUMNS:
     [[(col("n_adults")) + (col("n_children"))].alias("total_guests")] 
      FILTER [(col("status")) == ("Not_Canceled")]
      FROM
        DF ["ID", "n_adults", "n_children", "weekend_nights", ...]; PROJECT["year", "month", "date", "n_adults", ...] 6/19 COLUMNS


In [58]:
optimized_daily_guests = logs_hotel.select(["year", "month", "date", "n_adults", "n_children", "status"]
).filter(pl.col("status") == "Not_Canceled").with_columns(
    (pl.col("n_adults") + pl.col("n_children")).alias("total_guests")
).group_by("year", "month", "date").agg(
    pl.sum("total_guests").alias("daily_total_guests")
).sort("daily_total_guests", descending=True)

print(optimized_daily_guests)

shape: (538, 4)
┌──────┬───────┬──────┬────────────────────┐
│ year ┆ month ┆ date ┆ daily_total_guests │
│ ---  ┆ ---   ┆ ---  ┆ ---                │
│ i64  ┆ i64   ┆ i64  ┆ i64                │
╞══════╪═══════╪══════╪════════════════════╡
│ 2017 ┆ 8     ┆ 14   ┆ 365                │
│ 2018 ┆ 10    ┆ 13   ┆ 352                │
│ 2017 ┆ 9     ┆ 18   ┆ 345                │
│ 2018 ┆ 12    ┆ 27   ┆ 263                │
│ 2018 ┆ 12    ┆ 8    ┆ 259                │
│ …    ┆ …     ┆ …    ┆ …                  │
│ 2017 ┆ 7     ┆ 20   ┆ 4                  │
│ 2017 ┆ 7     ┆ 9    ┆ 2                  │
│ 2017 ┆ 7     ┆ 3    ┆ 2                  │
│ 2017 ┆ 7     ┆ 13   ┆ 2                  │
│ 2017 ┆ 7     ┆ 22   ┆ 1                  │
└──────┴───────┴──────┴────────────────────┘


Во всех пунктах планы запросов отличались лишь тем, что ленивый режим всегда сразу выбирает только необходимые столбцы. Это же учитывалось и в попытках оптимизировать запросы.